# Phase 3: Unsupervised Learning

## 🎯 Learning Objectives

By the end of this notebook, you will:

- ✅ Understand clustering algorithms
- ✅ Apply K-Means, Hierarchical, and DBSCAN clustering
- ✅ Use dimensionality reduction (PCA, t-SNE)
- ✅ Identify patterns in unlabeled data
- ✅ Visualize high-dimensional data

**Time Required:** 1-2 weeks  
**Difficulty:** Intermediate  
**Prerequisites:** Phases 0-2 completed

## What is Unsupervised Learning?

**Key Difference from Supervised:** NO LABELS!

**Goal:** Find hidden patterns, structures, or groups in data.

**Use Cases:**
- 👥 Customer segmentation
- 🔍 Anomaly detection (fraud)
- 📄 Topic modeling in documents
- 🧬 Gene clustering
- 🎵 Music recommendation

In [ ]:
# Import all required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Clustering algorithms
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN

# Dimensionality reduction
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Preprocessing and metrics
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples

# Datasets
from sklearn.datasets import make_blobs, make_moons, load_iris

# Settings
plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

print("✅ All libraries imported successfully!")

---

# Part A: Clustering

## 3.1 K-Means Clustering

### Concept
Group similar data points into K clusters.

**Algorithm:**
1. Choose K (number of clusters)
2. Randomly place K centroids
3. Assign each point to nearest centroid
4. Move centroids to center of assigned points
5. Repeat steps 3-4 until convergence

In [ ]:
# K-Means: Simple Example

# Generate sample data
X, y_true = make_blobs(n_samples=300, centers=4, n_features=2,
                       cluster_std=0.6, random_state=42)

# Apply K-Means
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
labels = kmeans.fit_predict(X)
centers = kmeans.cluster_centers_

# Visualize
plt.figure(figsize=(12, 5))

# Before clustering
plt.subplot(1, 2, 1)
plt.scatter(X[:, 0], X[:, 1], s=50, alpha=0.6)
plt.title('Before Clustering', fontsize=13, fontweight='bold')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.grid(True, alpha=0.3)

# After clustering
plt.subplot(1, 2, 2)
plt.scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', s=50, alpha=0.6)
plt.scatter(centers[:, 0], centers[:, 1], c='red', s=200, marker='X',
            edgecolors='black', linewidths=2, label='Centroids')
plt.title('After K-Means (K=4)', fontsize=13, fontweight='bold')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Cluster centers:\n{centers}")
print(f"\nInertia (sum of squared distances): {kmeans.inertia_:.2f}")

In [ ]:
# Real Example: Customer Segmentation by Spending

# Generate customer data
np.random.seed(42)
n_customers = 300

# Create different customer segments
# Segment 1: Young, low income, high spending score (students)
seg1 = np.column_stack([
    np.random.normal(25, 5, 75),   # Age
    np.random.normal(25000, 5000, 75),  # Income
    np.random.normal(70, 15, 75)   # Spending score
])

# Segment 2: Middle-aged, medium income, medium spending
seg2 = np.column_stack([
    np.random.normal(40, 8, 75),
    np.random.normal(55000, 10000, 75),
    np.random.normal(50, 12, 75)
])

# Segment 3: Older, high income, low spending (savers)
seg3 = np.column_stack([
    np.random.normal(55, 7, 75),
    np.random.normal(90000, 15000, 75),
    np.random.normal(30, 10, 75)
])

# Segment 4: Middle-aged, high income, high spending (affluent)
seg4 = np.column_stack([
    np.random.normal(45, 10, 75),
    np.random.normal(85000, 12000, 75),
    np.random.normal(80, 10, 75)
])

# Combine
data = np.vstack([seg1, seg2, seg3, seg4])
data = np.clip(data, 0, None)  # Ensure non-negative

df = pd.DataFrame(data, columns=['Age', 'Annual_Income', 'Spending_Score'])

print("Customer Dataset:")
print(df.head(10))
print(f"\nShape: {df.shape}")

In [ ]:
# Scale features for K-Means
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

# Apply K-Means
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

# Visualize 2D projections
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Age vs Income
scatter1 = axes[0].scatter(df['Age'], df['Annual_Income'], 
                           c=df['Cluster'], cmap='viridis', s=50, alpha=0.6)
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Annual Income ($)')
axes[0].set_title('Age vs Income', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Income vs Spending
scatter2 = axes[1].scatter(df['Annual_Income'], df['Spending_Score'], 
                           c=df['Cluster'], cmap='viridis', s=50, alpha=0.6)
axes[1].set_xlabel('Annual Income ($)')
axes[1].set_ylabel('Spending Score')
axes[1].set_title('Income vs Spending', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

# Age vs Spending
scatter3 = axes[2].scatter(df['Age'], df['Spending_Score'], 
                           c=df['Cluster'], cmap='viridis', s=50, alpha=0.6)
axes[2].set_xlabel('Age')
axes[2].set_ylabel('Spending Score')
axes[2].set_title('Age vs Spending', fontsize=12, fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.colorbar(scatter1, ax=axes, label='Cluster')
plt.suptitle('Customer Segmentation Results', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Cluster Analysis

cluster_summary = df.groupby('Cluster').agg({
    'Age': ['mean', 'std', 'count'],
    'Annual_Income': ['mean', 'std'],
    'Spending_Score': ['mean', 'std']
}).round(2)

print("=== Cluster Summary ===")
print(cluster_summary)

# Interpret clusters
print("\n=== Cluster Interpretation ===")
for cluster in range(4):
    cluster_data = df[df['Cluster'] == cluster]
    print(f"\n📊 Cluster {cluster} ({len(cluster_data)} customers):")
    print(f"   Avg Age: {cluster_data['Age'].mean():.1f} years")
    print(f"   Avg Income: ${cluster_data['Annual_Income'].mean():,.0f}")
    print(f"   Avg Spending: {cluster_data['Spending_Score'].mean():.1f}/100")

### Choosing Optimal K: Elbow Method & Silhouette Score

In [ ]:
# Elbow Method and Silhouette Score

# Generate fresh data
X, _ = make_blobs(n_samples=300, centers=4, n_features=2, random_state=42)

# Try different K values
k_range = range(2, 11)
inertias = []
silhouette_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(X, kmeans.labels_))

# Plot both methods
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Elbow method
axes[0].plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Clusters (K)', fontsize=12)
axes[0].set_ylabel('Inertia', fontsize=12)
axes[0].set_title('Elbow Method', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].axvline(x=4, color='red', linestyle='--', label='Elbow at K=4')
axes[0].legend()

# Silhouette score
axes[1].plot(k_range, silhouette_scores, 'go-', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Clusters (K)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('Silhouette Score Method', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

best_k = k_range[np.argmax(silhouette_scores)]
axes[1].axvline(x=best_k, color='red', linestyle='--', label=f'Best K={best_k}')
axes[1].legend()

plt.tight_layout()
plt.show()

print("Interpretation:")
print(f"- Elbow Method: Look for the 'bend' in the curve")
print(f"- Silhouette Score: Higher is better (K={best_k} has score={max(silhouette_scores):.3f})")

---

## 3.2 Hierarchical Clustering

### Concept
Build a tree (dendrogram) of clusters by merging or splitting.

**Two Types:**
- **Agglomerative (bottom-up):** Start with each point as cluster, merge
- **Divisive (top-down):** Start with all points in one cluster, split

In [ ]:
# Hierarchical Clustering with Dendrogram

from scipy.cluster.hierarchy import dendrogram, linkage

# Generate data
X, _ = make_blobs(n_samples=50, centers=3, n_features=2, random_state=42)

# Compute linkage
Z = linkage(X, method='ward')

# Plot dendrogram
plt.figure(figsize=(14, 6))
dendrogram(Z, truncate_mode='lastp', p=30, leaf_rotation=90, 
           leaf_font_size=10, show_contracted=True)
plt.xlabel('Sample Index', fontsize=12)
plt.ylabel('Distance', fontsize=12)
plt.title('Hierarchical Clustering Dendrogram', fontsize=14, fontweight='bold')
plt.axhline(y=50, color='r', linestyle='--', label='Cut for 3 clusters')
plt.legend()
plt.tight_layout()
plt.show()

print("Dendrogram shows how clusters merge at different distance levels.")
print("Cut horizontally to get desired number of clusters.")

In [ ]:
# Apply Agglomerative Clustering

# Generate more data
X, y_true = make_blobs(n_samples=300, centers=4, n_features=2, 
                       cluster_std=0.6, random_state=42)

# Different linkage methods
linkage_methods = ['ward', 'complete', 'average', 'single']
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.ravel()

for ax, method in zip(axes, linkage_methods):
    # Fit
    agg = AgglomerativeClustering(n_clusters=4, linkage=method)
    labels = agg.fit_predict(X)
    
    # Silhouette score
    score = silhouette_score(X, labels)
    
    # Plot
    ax.scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', s=50, alpha=0.6)
    ax.set_title(f'Linkage: {method}\nSilhouette: {score:.3f}', 
                fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.suptitle('Hierarchical Clustering: Different Linkage Methods', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nLinkage Methods:")
print("- Ward: Minimizes variance (most common)")
print("- Complete: Maximum distance between clusters")
print("- Average: Average distance between clusters")
print("- Single: Minimum distance (can create chain-like clusters)")

---

## 3.3 DBSCAN (Density-Based Clustering)

### Concept
Groups together points that are closely packed (high density).

**Key Parameters:**
- **eps:** Maximum distance between two samples to be in same neighborhood
- **min_samples:** Minimum points to form a dense region

**Advantages:**
- Finds arbitrarily shaped clusters
- Detects outliers (noise)
- No need to specify K

In [ ]:
# DBSCAN: Non-spherical Clusters

# Generate moon-shaped data (non-linear clusters)
X, y_true = make_moons(n_samples=300, noise=0.05, random_state=42)

# Compare K-Means vs DBSCAN
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Original data
axes[0].scatter(X[:, 0], X[:, 1], s=50, alpha=0.6)
axes[0].set_title('Original Data', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# K-Means (fails with non-spherical)
kmeans = KMeans(n_clusters=2, random_state=42)
labels_kmeans = kmeans.fit_predict(X)
axes[1].scatter(X[:, 0], X[:, 1], c=labels_kmeans, cmap='viridis', s=50, alpha=0.6)
axes[1].set_title(f'K-Means (K=2)\n❌ Fails!', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

# DBSCAN (works!)
dbscan = DBSCAN(eps=0.2, min_samples=5)
labels_dbscan = dbscan.fit_predict(X)
axes[2].scatter(X[:, 0], X[:, 1], c=labels_dbscan, cmap='viridis', s=50, alpha=0.6)
axes[2].set_title(f'DBSCAN\n✅ Works!', fontsize=12, fontweight='bold')
axes[2].grid(True, alpha=0.3)

plt.suptitle('K-Means vs DBSCAN on Moon-Shaped Data', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

n_clusters = len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
n_noise = list(labels_dbscan).count(-1)
print(f"DBSCAN found {n_clusters} clusters and {n_noise} noise points")

In [ ]:
# Effect of DBSCAN Parameters

# Generate data with noise
X, _ = make_blobs(n_samples=300, centers=3, n_features=2, 
                  cluster_std=0.5, random_state=42)
# Add noise
noise = np.random.uniform(-4, 4, (30, 2))
X = np.vstack([X, noise])

# Different eps values
eps_values = [0.2, 0.3, 0.5, 0.8]
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.ravel()

for ax, eps in zip(axes, eps_values):
    dbscan = DBSCAN(eps=eps, min_samples=5)
    labels = dbscan.fit_predict(X)
    
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = list(labels).count(-1)
    
    # Plot (noise points in black)
    colors = [plt.cm.viridis(l / max(labels.max(), 1)) if l >= 0 else 'black' for l in labels]
    ax.scatter(X[:, 0], X[:, 1], c=colors, s=50, alpha=0.6)
    ax.set_title(f'eps={eps}\nClusters: {n_clusters}, Noise: {n_noise}', 
                fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.suptitle('DBSCAN: Effect of eps Parameter\n(Black points = noise)', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

# Part B: Dimensionality Reduction

## 3.4 Principal Component Analysis (PCA)

### Concept
Reduce dimensions while preserving maximum variance.

**Use Cases:**
- Visualization of high-dimensional data
- Feature reduction before ML
- Noise reduction
- Speed up training

In [ ]:
# PCA on Iris Dataset

# Load iris (4 features)
iris = load_iris()
X = iris.data
y = iris.target

print(f"Original data shape: {X.shape}")
print(f"Features: {iris.feature_names}")

# Scale data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply PCA (reduce to 2D)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"\nReduced data shape: {X_pca.shape}")
print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total variance explained: {sum(pca.explained_variance_ratio_)*100:.1f}%")

In [ ]:
# Visualize PCA Results

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 2D PCA projection
scatter = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='viridis', 
                          s=50, alpha=0.6, edgecolors='black', linewidths=0.5)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=11)
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=11)
axes[0].set_title('Iris Dataset: PCA Projection', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Add legend
handles = [plt.scatter([], [], c=plt.cm.viridis(i/2), s=50) 
           for i in range(3)]
axes[0].legend(handles, iris.target_names, title='Species')

# Explained variance
pca_full = PCA().fit(X_scaled)
cumulative_var = np.cumsum(pca_full.explained_variance_ratio_)

axes[1].bar(range(1, 5), pca_full.explained_variance_ratio_, 
            alpha=0.7, label='Individual')
axes[1].plot(range(1, 5), cumulative_var, 'ro-', linewidth=2, 
             markersize=8, label='Cumulative')
axes[1].axhline(y=0.95, color='g', linestyle='--', label='95% threshold')
axes[1].set_xlabel('Principal Component', fontsize=11)
axes[1].set_ylabel('Explained Variance Ratio', fontsize=11)
axes[1].set_title('Variance Explained by Each PC', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nWith just 2 components, we explain {cumulative_var[1]*100:.1f}% of the variance!")

In [ ]:
# Feature Importance in PCA

# Get the loadings
pca_2d = PCA(n_components=2)
pca_2d.fit(X_scaled)

loadings = pd.DataFrame(
    pca_2d.components_.T,
    columns=['PC1', 'PC2'],
    index=iris.feature_names
)

print("=== PCA Loadings (Feature Importance) ===")
print(loadings.round(3))

# Visualize loadings
plt.figure(figsize=(10, 8))

# Plot data points
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='viridis', s=30, alpha=0.3)

# Plot loading vectors
for i, feature in enumerate(iris.feature_names):
    plt.arrow(0, 0, loadings.iloc[i, 0]*3, loadings.iloc[i, 1]*3,
              head_width=0.1, head_length=0.1, fc='red', ec='red')
    plt.text(loadings.iloc[i, 0]*3.2, loadings.iloc[i, 1]*3.2, 
             feature, fontsize=10, color='red')

plt.xlabel('PC1', fontsize=12)
plt.ylabel('PC2', fontsize=12)
plt.title('PCA Biplot: Data Points and Feature Vectors', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- Arrows show which features contribute to each PC")
print("- Longer arrows = more important features")
print("- Similar direction = positively correlated features")

---

## 3.5 t-SNE (t-Distributed Stochastic Neighbor Embedding)

### Concept
Non-linear dimensionality reduction, great for visualization.

**Differences from PCA:**
- PCA: Preserves global structure (linear)
- t-SNE: Preserves local structure (non-linear)

In [ ]:
# Compare PCA vs t-SNE

# Use iris data
X_scaled = StandardScaler().fit_transform(iris.data)

# PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_scaled)

# Compare
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PCA
axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='viridis', s=50, alpha=0.6)
axes[0].set_title('PCA', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Component 1')
axes[0].set_ylabel('Component 2')
axes[0].grid(True, alpha=0.3)

# t-SNE
axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=y, cmap='viridis', s=50, alpha=0.6)
axes[1].set_title('t-SNE', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Component 1')
axes[1].set_ylabel('Component 2')
axes[1].grid(True, alpha=0.3)

plt.suptitle('PCA vs t-SNE on Iris Dataset', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Observations:")
print("- t-SNE separates clusters more clearly")
print("- t-SNE is better for visualization")
print("- PCA is faster and more reproducible")

In [ ]:
# Effect of Perplexity in t-SNE

perplexities = [5, 20, 50, 100]
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.ravel()

for ax, perp in zip(axes, perplexities):
    tsne = TSNE(n_components=2, perplexity=perp, random_state=42)
    X_tsne = tsne.fit_transform(X_scaled)
    
    ax.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y, cmap='viridis', s=50, alpha=0.6)
    ax.set_title(f'Perplexity = {perp}', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.suptitle('t-SNE: Effect of Perplexity', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Perplexity Guidelines:")
print("- Low (5-10): Focus on very local structure")
print("- Medium (30-50): Balanced (recommended)")
print("- High (100+): Focus on global structure")

---

## 3.6 Combining Clustering with Dimensionality Reduction

In [ ]:
# Complete Pipeline: PCA + K-Means

# Generate high-dimensional data
from sklearn.datasets import make_blobs
X, y_true = make_blobs(n_samples=500, n_features=10, centers=5, random_state=42)

print(f"Original data shape: {X.shape}")

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA for visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Cluster in original space
kmeans = KMeans(n_clusters=5, random_state=42)
clusters = kmeans.fit_predict(X_scaled)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# True labels
axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=y_true, cmap='viridis', s=30, alpha=0.6)
axes[0].set_title('True Labels (PCA Projection)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].grid(True, alpha=0.3)

# K-Means clusters
axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, cmap='viridis', s=30, alpha=0.6)
axes[1].set_title('K-Means Clusters (PCA Projection)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Evaluate clustering
from sklearn.metrics import adjusted_rand_score
ari = adjusted_rand_score(y_true, clusters)
silhouette = silhouette_score(X_scaled, clusters)

print(f"\nClustering Quality:")
print(f"- Adjusted Rand Index: {ari:.3f} (1.0 = perfect)")
print(f"- Silhouette Score: {silhouette:.3f}")

---

## 📝 Practice Exercises

### Exercise 1: Customer Segmentation

In [ ]:
# Exercise 1: Perform customer segmentation

# Create customer data
np.random.seed(42)
n = 400

data = pd.DataFrame({
    'Recency': np.random.exponential(30, n),  # Days since last purchase
    'Frequency': np.random.poisson(5, n),      # Number of purchases
    'Monetary': np.random.exponential(100, n)  # Total spending
})

print("Customer RFM Data:")
print(data.head(10))

# TODO:
# 1. Scale the features
# 2. Use the Elbow method to find optimal K
# 3. Apply K-Means clustering
# 4. Analyze each customer segment
# 5. Visualize using PCA

print("\nComplete the exercise!")

---

## ✅ Phase Completion Checklist

- [ ] Apply K-Means clustering with optimal K selection
- [ ] Understand and use Hierarchical clustering
- [ ] Apply DBSCAN for non-spherical clusters
- [ ] Use PCA for dimensionality reduction
- [ ] Apply t-SNE for visualization
- [ ] Combine clustering with dimensionality reduction
- [ ] Interpret clustering results

---

## 🎯 Key Takeaways

1. **K-Means**: Fast, simple, but needs K specified; spherical clusters only
2. **Hierarchical**: Visual hierarchy, no K needed, computationally expensive
3. **DBSCAN**: Finds arbitrary shapes, detects outliers, needs eps tuning
4. **PCA**: Linear, fast, interpretable, preserves global structure
5. **t-SNE**: Non-linear, great for visualization, preserves local structure
6. **Always scale features** before clustering and dimensionality reduction!

---

## 📚 Next Steps

👉 **[Phase-4-Feature-Engineering.ipynb](Phase-4-Feature-Engineering.ipynb)** - Master data preprocessing!

This is where the real ML work happens:
- Handling missing data
- Dealing with outliers
- Feature encoding and scaling
- Feature creation and selection

**Happy Learning! 🚀**